## Homework 3

In the third homework we had the opportunity to learn about how to leverage google cloud storage (buckets) and Big Query on Google Cloud to analyse our datasets and query them with SQL

#### Preparation

For creating a bucket on Google Cloud I used Terraform. The exact details can be found at [this terraform file](main.tf). I used variables to make the main terraform file easier to read. The meanings can be found in [the variable file](variables.tf)

The next step used Python to transfer the yellow tripdata from 2024-01 to 2024-06. The data was first put into the bucket created in the previous step. Then I created a dataset in Big Query, where I transferred the data from buckets into tables.

The code is in [this Python file](dataimport.py)


Then an external and a regular table were created

**Creating external table**

In [ ]:
CREATE OR REPLACE EXTERNAL TABLE `natural-oath-484714-i1.ny_taxi.yellow_taxi_external`
OPTIONS (
  format = 'PARQUET',
  uris = ['gs://natural-oath-484714-i1-terra-bucket/yellow_tripdata_*.parquet']
);

**Regular Table Creation**

In [ ]:
CREATE OR REPLACE TABLE `natural-oath-484714-i1.ny_taxi.yellow_taxi`
AS
SELECT *
FROM `natural-oath-484714-i1.ny_taxi.yellow_taxi_external`;

<hr>

## Questions

**Question 1**

What is count of records for the 2024 Yellow Taxi Data?


<div style = "border: 2px solid white; padding: 2px; text-align: center">
20332093
</div>

**Question 2**

Write a query to count the distinct number of PULocationIDs for the entire dataset on both the tables.




Regular Table

In [ ]:
SELECT COUNT(DISTINCT PULocationID)
FROM `ny_taxi.yellow_taxi`;



<div style = "border: 2px solid white; padding: 2px; text-align: center">
155.12 MB
</div>

External Table

In [ ]:
SELECT COUNT(DISTINCT PULocationID)
FROM `ny_taxi.yellow_taxi_external`;

<div style = "border: 2px solid white; padding: 2px; text-align: center">
0 MB
</div>

**Question 3**

Write a query to retrieve the PULocationID from the table (not the external table) in BigQuery. Now write a query to retrieve the PULocationID and DOLocationID on the same table.

Why are the estimated number of Bytes different?

Queries

In [ ]:
SELECT PULocationID FROM `ny_taxi.yellow_taxi`

SELECT PULocationID, DOLocationID FROM `ny_taxi.yellow_taxi`

Selecting one column


<div style = "border: 2px solid white; padding: 2px; text-align: center">
155.12 MB
</div>


Selecting both columns


<div style = "border: 2px solid white; padding: 2px; text-align: center">
310.24 MB 
</div>

BigQuery is a columnar database, and it only scans the specific columns requested in the query. Querying two columns (PULocationID, DOLocationID) requires reading more data than querying one column (PULocationID), leading to a higher estimated number of bytes processed.

**Question 4** 

How many records have a fare_amount of 0?

In [ ]:
SELECT COUNT(*) fare_amount FROM `ny_taxi.yellow_taxi`
WHERE fare_amount = 0

<div style = "border: 2px solid white; padding: 2px; text-align: center">
8333 
</div>

**Question 5**

What is the best strategy to make an optimized table in Big Query if your query will always filter based on tpep_dropoff_datetime and order the results by VendorID (Create a new table with this strategy)

Partition by tpep_dropoff_datetime and Cluster on VendorID is the correct answer


Clustering works best for columns on which you can use ORDER BY, which is the case for VendorID. Partitioning works best for tpep_dropoff_datetime because this column is oftne used for filtering

In [ ]:
CREATE OR REPLACE TABLE `ny_taxi.yellow_taxi_optimized`
PARTITION BY DATE(tpep_dropoff_datetime)
CLUSTER BY VendorID
AS
SELECT *
FROM `ny_taxi.yellow_taxi`;


**Question 6**

Write a query to retrieve the distinct VendorIDs between tpep_dropoff_datetime 2024-03-01 and 2024-03-15 (inclusive)

Use the materialized table you created earlier in your from clause and note the estimated bytes. Now change the table in the from clause to the partitioned table you created for question 5 and note the estimated bytes processed. What are these values?

In [ ]:
SELECT DISTINCT VendorID, tpep_dropoff_datetime
FROM `ny_taxi.yellow_taxi`
WHERE tpep_dropoff_datetime BETWEEN 
      TIMESTAMP('2024-03-01') 
  AND TIMESTAMP('2024-03-15');


For regular table



<div style = "border: 2px solid white; padding: 2px; text-align: center">
310.24 MB
</div>

For partitioned and clustered table

<div style = "border: 2px solid white; padding: 2px; text-align: center">
26.84 MB
</div>

**Question 7**

Where is the data stored in the External Table you created?


<div style = "border: 2px solid white; padding: 2px; text-align: center">
GCP Bucket
</div>

**Question 8** 

It is best practice in Big Query to always cluster your data:


<div style = "border: 2px solid white; padding: 2px; text-align: center">
False
</div>

**Question 9**

No Points: Write a SELECT count(*) query FROM the materialized table you created. How many bytes does it estimate will be read? Why?


In [ ]:
SELECT COUNT(*) FROM `ny_taxi.yellow_taxi`

0B